# 04 - Ensemble Optimization

**Purpose.** Examine whether the optimized ensemble improves over a simple equal-weight ensemble, and inspect what the optimizer actually learned.

**Research integrity rule.** The equal-weight ensemble is a mandatory baseline. Optimization is only meaningful if it beats a transparent baseline on held-out test data.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path('..').resolve()
EXPERIMENT = PROJECT / 'artifacts' / 'paper_2022_idc' / 'experiment.json'
if not EXPERIMENT.exists():
    EXPERIMENT = PROJECT / 'artifacts' / 'local_dataset_smoke' / 'experiment.json'

print(EXPERIMENT)
print('exists:', EXPERIMENT.exists())

In [ ]:
if not EXPERIMENT.exists():
    print('Run an experiment first. Suggested quick command:')
    print('uv run dcpgann-train data/idc --config configs/paper_2022_idc.json --epochs 1 --backbones simple_cnn --output artifacts/local_dataset_smoke')
else:
    report = json.loads(EXPERIMENT.read_text())
    print('optimized method:', report['optimized_ensemble_validation']['method'])

## Equal Weight vs Optimized Ensemble

In [ ]:
if EXPERIMENT.exists():
    comparison = pd.DataFrame({
        'equal_weight_test': report['equal_weight_ensemble_test_metrics'],
        'optimized_test': report['optimized_ensemble_test_metrics'],
    })
    display(comparison.loc[['accuracy', 'balanced_accuracy', 'precision', 'recall_sensitivity', 'specificity', 'f1', 'roc_auc']])
    delta = comparison['optimized_test'] - comparison['equal_weight_test']
    print('Delta balanced accuracy:', float(delta.loc['balanced_accuracy']))

## Optimizer Metadata

For weighted ensembles, inspect weights. For Cartesian-program ensembles, inspect the evolved graph genome.

In [ ]:
if EXPERIMENT.exists():
    opt = report['optimized_ensemble_validation']
    print('method:', opt['method'])
    print('validation score:', opt['score'])
    if opt['weights']:
        weights = pd.Series(opt['weights'], index=report['backbones'], name='weight')
        display(weights.to_frame())
    else:
        genome = opt['metadata'].get('genome', {})
        display(pd.DataFrame({k: v for k, v in genome.items() if isinstance(v, list)}))
        print('output source:', genome.get('output'))

## Cartesian Program Search History

In [ ]:
if EXPERIMENT.exists():
    history = report['optimized_ensemble_validation']['metadata'].get('history', [])
    if history:
        hist = pd.DataFrame(history)
        ax = hist.plot(x='generation', y='best_score', legend=False, figsize=(7, 3), title='CGP validation score')
        ax.set_ylabel(report['ensemble_config']['metric'])
        plt.show()
    else:
        print('No CGP history available; likely a weighted-logit run.')